# Kirill 3 Diya — analysis following `014_testing.ipynb`

Uses the cell, medium and optical-power maps from `plate_reader_analysis_diya_kirill_3.py` and the three `26-09-03 Kirill 3_diya_extracted_*.csv` files in `Yuhang/Data`.

This is one plate: columns 1–6 receive red light (2.8 in the source map), and columns 7–12 green light (0.28). Samples and blanks are grouped by both label and light; red and green wells are never pooled.

Run all cells in order. Edit **Settings** to select channels, samples, lights, plot types and exports. By default, both refreshed (`_D`) and stationary (`_O`) samples are shown. `K` denotes single plasmid and `N` double plasmid control. Suffixes 1–3 denote biological replicates, each with three technical wells per light. Biological replicates remain separate curves.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.embed import file_html
from IPython.display import HTML, display
from bokeh.resources import CDN
import html as html_lib

# Works when launched from the repository root or any directory beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Yuhang/plate_reader_analysis_diya_kirill_3.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Launch this notebook from within the CcaSolitaiRe repository.')
DATA_DIR = ROOT / 'Yuhang/Data'

## Settings

Blank correction uses the corresponding BK or BCS control, matched to growth state and light:

| Samples | Control | Red blank wells | Green blank wells |
|---|---|---|---|
| K_D_1–3 | BK_D | H1–H3 | H7–H9 |
| K_O_1–3 | BK_O | H4–H6 | H10–H12 |
| N_D_1–3 | BCS_D | G1–G3 | G7–G9 |
| N_O_1–3 | BCS_O | G4–G6 | G10–G12 |

This extends the incubator's confirmed K→BK and N→BCS assignments to the corresponding stationary controls. Blank wells are averaged per time point and channel. `BLANK_LABEL_BY_FAMILY` defines the assignments; `BLANK_WELLS` optionally selects a subset of each control's wells. Wells labelled `media` are not used.

`EXCLUDED_MEASUREMENTS` masks individual sample readings before averaging. Keys are `(light, well)` and values are zero-based reading indices. Summary plots omit the initial reading by default, as in `014_testing.ipynb`; set `SUMMARY_START_INDEX = 0` to include it. Replicate and blank plots show all readings.


In [2]:
LIGHTS = ['red', 'green']                 # Or ['green'] / ['red']; light regions of ONE plate
PROBE_CHANNEL = 'GFP 395nm'              # Or 'GFP 488nm'
SELECTED_SAMPLES = [f'{family}_{rep}' for family in ['K_D', 'N_D']#, 'K_O', 'N_O']
                    for rep in [1, 2, 3]]
PLOTS = ['replicates', 'blanks', 'summary']
SUMMARY_METRICS = ['normalized_fluorescence', 'od600']
SUMMARY_START_INDEX = 1                  # Set 0 to include the initial reading
Y_RANGES = {'normalized_fluorescence': None, 'od600': None}  # e.g. (0, 1000)
BLANK_LABEL_BY_FAMILY = {'K_D': 'BK_D', 'K_O': 'BK_O', 'N_D': 'BCS_D', 'N_O': 'BCS_O'}
BLANK_WELLS = {}                         # e.g. {('green', 'BK_D'): ['H7', 'H8', 'H9']}
EXCLUDED_MEASUREMENTS = {}               # e.g. {('green', 'D7'): [0]}
SAVE_HTML = False
OUTPUT_DIR = ROOT / 'Yuhang/26-09-03 Kirill 3/figures_diya_014_analysis'

## Plate maps

Arrays and labels copied from the Diya script without importing its executable plotting code. Light labels derived from the per-well optical-power arrays.

In [3]:
NCOLS, NROWS = 12, 8
CELL_NAMES = {0: 'null', 1: 'KS_D', 2: 'KS_O', 3: 'N_D', 4: 'N_O', 5: 'BK_D', 20: 'BK_O', 6: 'BCS_D', 19: 'BCS_O', 7: 'K_D_1', 8: 'K_D_2', 9: 'K_D_3', 10: 'K_O_1', 11: 'K_O_2', 12: 'K_O_3', 13: 'N_D_1', 14: 'N_D_2', 15: 'N_D_3', 16: 'N_O_1', 17: 'N_O_2', 18: 'N_O_3'}

cell_map = np.array([
    #1   2   3   4   5   6   7   8   9    10  11  12
    [13, 13, 13, 16, 16, 16, 13, 13, 13,  16, 16, 16],  # Row A
    [14, 14, 14, 17, 17, 17, 14, 14, 14,  17, 17, 17],  # Row B
    [15, 15, 15, 18, 18, 18, 15, 15, 15,  18, 18, 18],  # Row C
    [7,   7,  7, 10, 10, 10,  7,  7,  7,  10, 10, 10],  # Row D
    [8,   8,  8, 11, 11, 11,  8,  8,  8,  11, 11, 11],  # Row E
    [9,   9,  9, 12, 12, 12,  9,  9,  9,  12, 12, 12],  # Row F
    [6,   6,  6, 19, 19, 19,  6,  6,  6,  19, 19, 19],  # Row G
    [5,   5,  5, 20, 20, 20,  5,  5,  5,  20, 20, 20],  # Row H
])

OPTICAL_POWER = np.array([
    # Channel 0 – Blue (unused, all zeros)
    np.zeros((NROWS, NCOLS)),

    # Channel 1 – Green light
    np.array([
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row A
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row B
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row C
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row D
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row E
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row F
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row G
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row H
]),

    # Channel 2 – Yellow-Green / White (unused, all zeros)
    np.zeros((NROWS, NCOLS)),

    # Channel 3 – Red light
    np.array([
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row A
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row B
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row C
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row D
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row E
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row F
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row G
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row H
]),
])


Gather all the information into a single plate_map dataframe

In [4]:
# get well labels and sample identities
plate_map = pd.DataFrame([
    {'well': f'{row}{c+1}', 'sample': CELL_NAMES[int(cell_map[r,c])],
     'row': row, 'column': c+1}
    for r, row in enumerate('ABCDEFGH') for c in range(12)
]).set_index('well')

# verify that the plate map has a unique index
assert len(plate_map) == 96 and plate_map.index.is_unique

# get green and red light intensities
plate_map['green_intensity'] = OPTICAL_POWER[1].reshape(-1)
plate_map['red_intensity'] = OPTICAL_POWER[3].reshape(-1)
# verify that all wells are lit EITHER red or green
red_only = plate_map.red_intensity.gt(0) & plate_map.green_intensity.eq(0)
green_only = plate_map.green_intensity.gt(0) & plate_map.red_intensity.eq(0)
if not (red_only | green_only).all():
    raise ValueError('Expected exclusively red or green illumination for every Diya well.')
plate_map['light'] = np.where(green_only, 'green', 'red')

# verify that all the selected samples are included in the plate map
unknown = set(SELECTED_SAMPLES) - set(plate_map['sample'])
if unknown:
    raise ValueError(f'Samples absent from this plate map: {unknown}')
def sample_family(sample):
    return sample.rsplit('_', 1)[0] if sample.rsplit('_', 1)[-1].isdigit() else sample

# get blanks
## match each sample to the corresponding blanks
sample_blank = {}
for sample in SELECTED_SAMPLES:
    family = sample_family(sample)
    if family not in BLANK_LABEL_BY_FAMILY:
        raise ValueError(f'Assign a blank control in BLANK_LABEL_BY_FAMILY for {family}')
    sample_blank[sample] = BLANK_LABEL_BY_FAMILY[family]
blank_labels = list(dict.fromkeys(sample_blank.values()))
## record blanks
blank_wells = {}
for light in LIGHTS:
    if light not in ('red', 'green'):
        raise ValueError(f'Unknown light: {light}')
    blank_wells[light] = {}
    for label in blank_labels:
        wells = list(BLANK_WELLS.get((light, label),
                     plate_map.index[(plate_map['sample'].eq(label) & plate_map['light'].eq(light))].tolist()))
        if not wells or len(set(wells)) != len(wells):
            raise ValueError(f'Blank wells for {light}, {label} must be nonempty and unique.')
        if not set(wells) <= set(plate_map.index[(plate_map['sample'].eq(label) & plate_map['light'].eq(light))]):
            raise ValueError(f'Blank wells for {label} must match both the control label and light in the map.')
        blank_wells[light][label] = wells

# verify that each sample has been tested in each light condition
def sample_indices(sample, light):
    return np.flatnonzero(plate_map['sample'].eq(sample) & plate_map['light'].eq(light))
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        if not len(sample_indices(sample, light)):
            raise ValueError(f'No wells for {sample}, {light}')

# show the plate map for verification
display(plate_map.pivot(index='row', columns='column', values='sample'))
display(plate_map.pivot(index='row', columns='column', values='light'))

# show how samples are matched to their blanks
for light, blanks in blank_wells.items():
    for label, wells in blanks.items():
        print(f'{light}: {label} blank wells: {", ".join(wells)}')


column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,N_D_1,N_D_1,N_D_1,N_O_1,N_O_1,N_O_1,N_D_1,N_D_1,N_D_1,N_O_1,N_O_1,N_O_1
B,N_D_2,N_D_2,N_D_2,N_O_2,N_O_2,N_O_2,N_D_2,N_D_2,N_D_2,N_O_2,N_O_2,N_O_2
C,N_D_3,N_D_3,N_D_3,N_O_3,N_O_3,N_O_3,N_D_3,N_D_3,N_D_3,N_O_3,N_O_3,N_O_3
D,K_D_1,K_D_1,K_D_1,K_O_1,K_O_1,K_O_1,K_D_1,K_D_1,K_D_1,K_O_1,K_O_1,K_O_1
E,K_D_2,K_D_2,K_D_2,K_O_2,K_O_2,K_O_2,K_D_2,K_D_2,K_D_2,K_O_2,K_O_2,K_O_2
F,K_D_3,K_D_3,K_D_3,K_O_3,K_O_3,K_O_3,K_D_3,K_D_3,K_D_3,K_O_3,K_O_3,K_O_3
G,BCS_D,BCS_D,BCS_D,BCS_O,BCS_O,BCS_O,BCS_D,BCS_D,BCS_D,BCS_O,BCS_O,BCS_O
H,BK_D,BK_D,BK_D,BK_O,BK_O,BK_O,BK_D,BK_D,BK_D,BK_O,BK_O,BK_O


column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,red,red,red,red,red,red,green,green,green,green,green,green
B,red,red,red,red,red,red,green,green,green,green,green,green
C,red,red,red,red,red,red,green,green,green,green,green,green
D,red,red,red,red,red,red,green,green,green,green,green,green
E,red,red,red,red,red,red,green,green,green,green,green,green
F,red,red,red,red,red,red,green,green,green,green,green,green
G,red,red,red,red,red,red,green,green,green,green,green,green
H,red,red,red,red,red,red,green,green,green,green,green,green


red: BK_D blank wells: H1, H2, H3
red: BCS_D blank wells: G1, G2, G3
green: BK_D blank wells: H7, H8, H9
green: BCS_D blank wells: G7, G8, G9


## Load readings and subtract blanks

OD and fluorescence are read minutes apart due to the speed at which our plate reader works - we take the OD measurement times as our reference. Both light regions share the same acquisition timestamps and use hours since the first OD reading of the Diya plate. The acquisition table shows the actual channel timestamps.

The data are processed according to the following formulae:

- `OD = max(raw OD − matched control mean OD, 0.005)`
- `fluorescence/OD = max((raw fluorescence − matched control mean fluorescence) / OD, 0)`

Correction and division happen per well, before averaging technical replicates. No path-length or volume correction is applied. Blank uncertainty is not propagated into the error bars.

We use population SD (`ddof=0`).  Missing/nonfinite input values raise an error rather than being silently included.

In [5]:
# initialise raw/processed measurements, measurement times, and blank measuremenys
raw, times, blanks, processed = {}, {}, {}, {}
acquisition_rows, qc_rows = [], []

# read measurements by light condition and by observation channel
for light in LIGHTS:
    raw[light] = {}
    for channel in ['OD600', PROBE_CHANNEL]:
        # read the file
        path = DATA_DIR / f'26-09-03 Kirill 3_diya_extracted_{channel}.csv'
        frame = pd.read_csv(path, index_col=0)

        # verify that there are records for all 96 well plates
        if not frame.index.is_unique or set(frame.index) != set(plate_map.index):
            raise ValueError(f'Expected exactly 96 unique wells in {path}')

        # get measurement times
        frame = frame.loc[plate_map.index].astype(float)
        stamps = pd.to_datetime(frame.columns, dayfirst=True, format='mixed')
        if not stamps.is_monotonic_increasing or not stamps.is_unique:
            raise ValueError(f'Timestamps must be unique and increasing: {path}')

        # veri that measurements are finite
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f'Non-finite measurements in {path}')

        # get raw measuremenet
        raw[light][channel] = frame

        # get the labels for acquired well
        for i, stamp in enumerate(stamps):
            acquisition_rows.append({'light': light, 'channel': channel, 'reading': i, 'timestamp': stamp})

        # it this is an OD measurement, set its time of acquisition for all measured variables
        if channel == 'OD600':
            times[light] = ((stamps - stamps[0]).total_seconds() / 3600).to_numpy()

    # get raw data for all samples
    od_raw = raw[light]['OD600'].to_numpy()
    flu_raw = raw[light][PROBE_CHANNEL].to_numpy()
    if od_raw.shape != flu_raw.shape:
        raise ValueError(f'Different numbers of OD and fluorescence data points for {light}')
    # get mean blank ODs
    blanks[light] = {
        label: {channel: frame.loc[wells].mean(axis=0).to_numpy()
                for channel, frame in raw[light].items()}
        for label, wells in blank_wells[light].items()
    }

    # Normalize measurements for selected samples; blanks/media/unselected samples get nans
    ## initialize with nans
    processed[light] = {metric: np.full_like(od_raw, np.nan)
                        for metric in ['od600', 'normalized_fluorescence']}
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        label = sample_blank[sample]
        blank = blanks[light][label]
        # normalize the OD and floor it
        od_delta = od_raw[idx] - blank['OD600']
        od = np.maximum(od_delta, 0.005)

        # normalize the fluorescence and floor it
        normalized = np.maximum((flu_raw[idx] - blank[PROBE_CHANNEL]) / od, 0)
        processed[light]['od600'][idx] = od
        processed[light]['normalized_fluorescence'][idx] = normalized
        qc_rows.append({'light': light, 'sample': sample, 'blank_control': label,
                        'blank_wells': ','.join(blank_wells[light][label]),
                        'technical_wells': len(idx),
                        'OD_values_floored': int((od_delta < 0.005).sum())})

# show the labels and time stamps for all acquired data
display(pd.DataFrame(acquisition_rows))
display(pd.DataFrame(qc_rows))

,light,channel,reading,timestamp
0,red,OD600,0,2026-09-03 09:00:35
1,red,OD600,1,2026-09-03 12:05:28
2,red,OD600,2,2026-09-03 15:04:42
3,red,OD600,3,2026-09-03 18:05:03
4,red,OD600,4,2026-09-03 21:09:43
5,red,OD600,5,2026-09-04 00:08:39
6,red,OD600,6,2026-09-04 09:14:11
7,red,OD600,7,2026-09-04 11:05:54
8,red,GFP 395nm,0,2026-09-03 09:02:00
9,red,GFP 395nm,1,2026-09-03 12:06:52


,light,sample,blank_control,blank_wells,technical_wells,OD_values_floored
0,red,K_D_1,BK_D,"H1,H2,H3",3,0
1,red,K_D_2,BK_D,"H1,H2,H3",3,1
2,red,K_D_3,BK_D,"H1,H2,H3",3,0
3,red,N_D_1,BCS_D,"G1,G2,G3",3,0
4,red,N_D_2,BCS_D,"G1,G2,G3",3,0
5,red,N_D_3,BCS_D,"G1,G2,G3",3,0
6,green,K_D_1,BK_D,"H7,H8,H9",3,1
7,green,K_D_2,BK_D,"H7,H8,H9",3,0
8,green,K_D_3,BK_D,"H7,H8,H9",3,1
9,green,N_D_1,BCS_D,"G7,G8,G9",3,2


Apply masks to exclude measurements

In [6]:
# Masks are applied to samples, not to the blank averages.
masks = {light: np.ones_like(processed[light]['od600'], dtype=bool) for light in LIGHTS}
for (light, well), indices in EXCLUDED_MEASUREMENTS.items():
    if light not in masks or well not in plate_map.index:
        raise ValueError(f'Unknown mask target: {light}, {well}')
    if plate_map.loc[well, 'light'] != light or plate_map.loc[well, 'sample'] not in SELECTED_SAMPLES:
        raise ValueError(f'Mask target must be a selected sample under {light}: {well}')
    if any(i < 0 or i >= len(times[light]) for i in indices):
        raise ValueError(f'Invalid reading index for {light}, {well}')
    masks[light][plate_map.index.get_loc(well), indices] = False

# Gather the data into a summary dataframe
summary_rows = []
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        for metric, values in processed[light].items():
            for t, hours in enumerate(times[light]):
                included = values[idx, t][masks[light][idx, t]]
                summary_rows.append(dict(light=light, sample=sample, metric=metric,
                    blank_control=sample_blank[sample], reading=t, hours=hours, n=len(included),
                    mean=included.mean() if len(included) else np.nan,
                    sd=included.std(ddof=0) if len(included) else np.nan))
summary = pd.DataFrame(summary_rows)
# show a representatie entry in the summary dataframe
display(summary.head(12))

,light,sample,metric,blank_control,reading,hours,n,mean,sd
0,red,K_D_1,od600,BK_D,0,0.000000,3,0.012867,0.000779
1,red,K_D_1,od600,BK_D,1,3.081389,3,0.234667,0.007035
2,red,K_D_1,od600,BK_D,2,6.068611,3,0.337633,0.000726
3,red,K_D_1,od600,BK_D,3,9.074444,3,0.376900,0.008437
4,red,K_D_1,od600,BK_D,4,12.152222,3,0.409533,0.008416
5,red,K_D_1,od600,BK_D,5,15.134444,3,0.444433,0.009045
6,red,K_D_1,od600,BK_D,6,24.226667,3,0.412200,0.015678
7,red,K_D_1,od600,BK_D,7,26.088611,3,0.417567,0.016879
8,red,K_D_1,normalized_fluorescence,BK_D,0,0.000000,3,161.420505,228.283068
9,red,K_D_1,normalized_fluorescence,BK_D,1,3.081389,3,0.000000,0.000000


## Plots

Replicate panels show each technical well plus its masked mean. Blank panels show raw selected blank wells and their mean, before OD flooring. Summary panels group `K_D`, `N_D`, etc., retain a separate curve for each biological replicate, and show ±1 SD across its technical wells. Click legend entries to hide curves. Bokeh's toolbar supports zoom, pan, reset and image saving; `SAVE_HTML` also writes interactive standalone HTML files.

Plot measurements:

In [7]:
# specify plotting styles
COLORS = {'red': 'crimson', 'green': 'darkgreen'}
DASHES = ['solid', 'dashed', 'dotted']
LABELS = {'od600': 'Blanked OD600',
          'normalized_fluorescence': f'{PROBE_CHANNEL} / OD600 (blanked, AU)'}

# figure creation function
def new_figure(title, ylabel, metric=None):
    options = {} if Y_RANGES.get(metric) is None else {'y_range': Y_RANGES[metric]}
    return figure(title=title, x_axis_label='Time since first OD reading, h',
                  y_axis_label=ylabel, width=400, height=300,
                  tools='pan,wheel_zoom,box_zoom,reset,save', **options)

#  show all plots after styling the legends
def emit(layout, name):
    from bokeh.models import Legend
    for legend in layout.select({'type': Legend}):
        legend.location = 'top_left'
        legend.background_fill_color = 'white'
        legend.background_fill_alpha = 1.0
        legend.border_line_color = 'black'
        legend.border_line_alpha = 1.0
    html_doc = file_html(layout, CDN, 'Plots')
    display(HTML(f'<iframe srcdoc="{html_lib.escape(html_doc)}" width="100%" height="700" style="border:none;"></iframe>'))
    if SAVE_HTML:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        channel_slug = PROBE_CHANNEL.replace(' ', '_')
        target = OUTPUT_DIR / f'{channel_slug}_{name}.html'
        target.write_text(file_html(layout, INLINE, name), encoding='utf-8')
        print(f'Saved {target}')

# plot data for different replicates
if 'replicates' in PLOTS:
    for metric in SUMMARY_METRICS:
        panels = []
        for sample in SELECTED_SAMPLES:
            for light in LIGHTS:
                p = new_figure(f'{sample}, {light}', LABELS[metric], metric)
                idx = sample_indices(sample, light)
                for j, row in enumerate(idx):
                    values = np.where(masks[light][row], processed[light][metric][row], np.nan)
                    p.line(times[light], values, color=COLORS[light], alpha=0.35,
                           line_dash=DASHES[j % 3], legend_label=plate_map.index[row])
                s = summary.query('light == @light and sample == @sample and metric == @metric')
                p.line(s.hours, s['mean'], color=COLORS[light], line_width=3, legend_label='Mean')
                p.legend.padding = 0
                p.legend.margin = 2
                p.legend.spacing = 0
                p.legend.click_policy = 'hide'
                panels.append(p)
        if panels:
            emit(gridplot(panels, ncols=len(LIGHTS)), f'replicates_{metric}')



/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Plot blanks:

In [8]:
if 'blanks' in PLOTS:
    panels = []
    for light in LIGHTS:
        for label, wells in blank_wells[light].items():
            for channel in ['OD600', PROBE_CHANNEL]:
                p = new_figure(f'{light}: {label} ({", ".join(wells)})', f'Raw {channel}')
                for well in wells:
                    p.line(times[light], raw[light][channel].loc[well].to_numpy(),
                           color=COLORS[light], alpha=0.35, legend_label=well)
                p.line(times[light], blanks[light][label][channel], color='black',
                       line_width=3, legend_label='Control mean')
                p.legend.padding = 0
                p.legend.margin = 2
                p.legend.spacing = 0
                p.legend.click_policy = 'hide'
                panels.append(p)
    emit(gridplot(panels, ncols=2), 'blanks')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [11]:
if 'summary' in PLOTS:
    families = list(dict.fromkeys(s.rsplit('_', 1)[0] if s.rsplit('_', 1)[-1].isdigit() else s
                                 for s in SELECTED_SAMPLES))
    for metric in SUMMARY_METRICS:
        panels = []
        for family in families:
            p = new_figure(f'{family}: technical means ± SD', LABELS[metric], metric)
            samples = [s for s in SELECTED_SAMPLES if s == family or s.rsplit('_', 1)[0] == family]
            for j, sample in enumerate(samples):
                for light in LIGHTS:
                    s = summary.query('light == @light and sample == @sample and metric == @metric').copy()
                    s = s[s.reading >= SUMMARY_START_INDEX]
                    label = f'{sample}, {light}'
                    p.segment(s.hours, s['mean'] - s.sd, s.hours, s['mean'] + s.sd,
                              color=COLORS[light], alpha=0.5, legend_label=label)
                    p.line(s.hours, s['mean'], color=COLORS[light], line_width=2,
                           line_dash=DASHES[j % 3], legend_label=label)
                    p.scatter(s.hours, s['mean'], color=COLORS[light],
                              marker=['plus', 'square', 'circle'][j % 3], size=7, legend_label=label)
            p.legend.padding = 0
            p.legend.margin = 2
            p.legend.spacing = 0
            p.legend.click_policy = 'hide'
            p.legend.label_text_font_size = '8pt'
            panels.append(p)


## Optional export

Set `EXPORT_SUMMARY = True` to save the displayed statistics, the plate map, and the selected blank means. `n` is the number of included technical wells at each time point. Changing blank wells, masks or channels requires rerunning the processing cells and plots.

In [14]:
EXPORT_SUMMARY = True
if EXPORT_SUMMARY:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_DIR / f'{PROBE_CHANNEL.replace(" ", "_")}_summary.csv', index=False)
    plate_map.to_csv(OUTPUT_DIR / 'plate_map.csv')
    pd.DataFrame([{'light': light, 'blank_control': label, 'channel': channel, 'reading': i,
                   'hours': times[light][i], 'blank_mean': value,
                   'blank_wells': ','.join(blank_wells[light][label])}
                  for light in LIGHTS for label, channels in blanks[light].items()
                  for channel, values in channels.items()
                  for i, value in enumerate(values)]).to_csv(OUTPUT_DIR / 'blank_means.csv', index=False)